# Limpeza e tratamento de dados de vendas de café

Este notebook realiza a leitura da tabela `dirty_cafe_sales_table`, converte os tipos das colunas, padroniza valores inválidos (como `ERROR`, `UNKNOWN` e `NULL`) para `NULL`, e então preenche os dados faltantes de forma incremental — preços unitários pelo cálculo de total/quantidade e pelo produto, itens pelo preço, total gasto pelo preço × quantidade, quantidade pelo total/preço — e, por fim, testa (e descarta) o preenchimento de datas inválidas via tendência do ID numérico e a associação entre método de pagamento, local de compra e produto.

O fluxo completo é o seguinte:

1. **Leitura e inspeção**: carrega a tabela `dirty_cafe_sales_table` e exibe informações gerais (quantidade de linhas, amostra, esquema e colunas).
2. **Criação do ID numérico**: extrai a parte numérica (sufixo) da coluna `transaction_id` (padrão `TXN_<id>`) com `regexp_extract` e cria a nova coluna inteira `numeric_id`.
3. **Conversão de tipos e padronização de inválidos**: a função `convert_column_types` converte cada coluna para o tipo apropriado (`string`, `int`, `double`, `date`, etc.) e, antes da conversão, substitui os valores inválidos (`ERROR`, `UNKNOWN`, `NULL`) por `NULL` em todas as colunas. Em seguida, `show_invalids` conta quantos `NULL`s restam por coluna.
4. **Funções auxiliares de visualização**: a função `plot_series` plota gráficos de séries temporais (ex.: `price_per_unit` ao longo de `transaction_date`, agrupados por `item`), marcando os pontos inválidos com asteriscos para facilitar a análise.
5. **Preenchimento de preços por cálculo**: a função `fill_price_by_calculation` preenche `price_per_unit` inválidos calculando `total_spent / quantity` quando ambos são válidos e a quantidade é diferente de zero.
6. **Preenchimento de preços pelo produto**: a função `fill_invalid_prices` preenche os `price_per_unit` inválidos remanescentes com base em um dicionário de preços conhecidos por item (`price_map`), uma vez que os preços são fixos ao longo do tempo, conforme confirmado pelo gráfico.
7. **Preenchimento de itens pelo preço**: a função `fill_items_by_prices` faz o caminho inverso — preenche os `item` inválidos com base no `price_per_unit`, usando um dicionário que mapeia preço → item. Nos casos de preços ambíguos (ex.: `3.0` pode ser Cake ou Juice; `4.0` pode ser Sandwich ou Smoothie), o item é preenchido como `Cake_or_Juice` ou `Sandwich_or_Smoothie`.
8. **Preenchimento do total gasto por cálculo**: a função `fill_total_spent_by_calculation` preenche `total_spent` inválidos calculando `price_per_unit * quantity` quando ambos são válidos.
9. **Preenchimento da quantidade por cálculo**: a função `fill_quantity_by_calculation` preenche `quantity` inválidos calculando `total_spent / price_per_unit` quando ambos são válidos e o preço é diferente de zero.
10. **Tentativa de preenchimento das datas inválidas**: as funções `plot_temporal_series` e `plot_temporal_series_by_average` plotam `numeric_id` ao longo de `transaction_date` (bruto e média diária) para verificar se o ID cresce monotonicamente com o tempo, o que permitiria inferir datas inválidas. Constatou-se que não há tendência nem agrupamento dos IDs por data, logo essa estratégia não é adequada para esses dados.
11. **Tentativa de associação entre método de pagamento, local de compra e produto**: a função `plot_histogram_by_combination` plota histogramas das combinações `location` × `payment_method`, `item` × `payment_method` e `item` × `location`, buscando dependências que permitissem preencher `payment_method` e `location` inválidos. Nenhuma relação foi observada, logo não é recomendado preencher essas colunas com base nessas variáveis (nem nos itens).
12. **Verificação final**: `show_invalids` resume quantos `NULL`s restam por coluna — `item`, `quantity`, `price_per_unit` e `total_spent` foram quase totalmente preenchidos; `payment_method`, `location` e `transaction_date` ainda exigem análise mais profunda (possivelmente com modelos de ML).
13. **Salvamento**: o DataFrame tratado é salvo como a tabela `workspace.my_database.clean_cafe_sales_table`.


# Abertura e exibição de informações gerais

In [0]:
data_path = 'workspace.my_database.dirty_cafe_sales_table'

df = spark.read.table(data_path)

df_len = df.count() # conta as linhas
print(f'DATA LENGHT: {df_len}')

df.show(5)
df.printSchema()

print('Columns:', df.columns)

# Criação do ID numérico

In [0]:
from pyspark.sql.functions import col, regexp_extract

# Como verificado na exploração dos dados, o ID de transação é composto por uma parte alfanumérica (prefixo TXN) e uma parte numérica com id único(sufixo)

# Aqui é extraída a parte numérica (sufixo) da coluna `transaction_id` (padrão TXN_<id>) e cria uma nova coluna inteira com o ID numérico.

df = df.withColumn(
    'numeric_id',
    regexp_extract(col('transaction_id'), r'TXN_(\d+)', 1)
)

# Reordena as colunas para que 'numeric_id' seja a primeira
cols = df.columns
cols.remove('numeric_id')
df = df.select('numeric_id', *cols)

df.show(5)

# Conversão de tipo das colunas

In [0]:
from pyspark.sql.functions import col, when, lit
from pyspark.sql.types import (
    StringType, IntegerType, DoubleType, FloatType,
    LongType, BooleanType, DateType, TimestampType,
    DecimalType, ShortType, ByteType, BinaryType
)


def convert_column_types(df, type_map, invalid_values=None):
    """
    Converte as colunas de um DataFrame Spark para os tipos especificados.

    Parâmetros:
        df             : DataFrame Spark.
        type_map       : dicionário no formato {coluna: tipo}, onde 'tipo' pode ser:
                        - uma string (ex.: 'string', 'int', 'double', 'date', 'timestamp', 'boolean', ...)
                        - um tipo nativo do pyspark.sql.types (ex.: IntegerType(), DoubleType(), ...)
                        - uma classe de tipo (ex.: IntegerType — será instanciada sem argumentos)
        invalid_values : lista de valores-string considerados inválidos (ex.: ['ERROR', 'UNKNOWN', 'NULL']).
                        Se fornecida, todos os valores dessa lista serão substituídos por NULL
                        em TODAS as colunas ANTES da conversão de tipo. O NULL é preservado
                        após a conversão, independente do tipo alvo.
                        Valores None (NULLs reais) são SEMPRE considerados inválidos,
                        automaticamente, independente de estarem ou não na lista.

    Retorna:
        DataFrame com as colunas convertidas para os tipos especificados.
    """
    # Mapeia strings para tipos nativos do pyspark.sql.types
    string_to_type = {
        'string': StringType(),
        'int': IntegerType(), 'integer': IntegerType(),
        'bigint': LongType(), 'long': LongType(),
        'smallint': ShortType(), 'short': ShortType(),
        'tinyint': ByteType(), 'byte': ByteType(),
        'double': DoubleType(),
        'float': FloatType(),
        'boolean': BooleanType(), 'bool': BooleanType(),
        'date': DateType(),
        'timestamp': TimestampType(),
        'binary': BinaryType(),
    }

    # Substitui os valores inválidos por NULL em todas as colunas ANTES da conversão.
    # None (NULL real) é SEMPRE tratado como inválido — isin() não captura NULLs
    # no PySpark, por isso .isNull() é sempre incluído na condição, além das strings.
    string_values = [v for v in (invalid_values or []) if v is not None]

    if string_values:
        for column_name in df.columns:
            condition = col(column_name).isNull() | col(column_name).isin(string_values)

            df = df.withColumn(
                column_name,
                when(condition, lit(None)).otherwise(col(column_name))
            )

    for column_name, target_type in type_map.items():
        if column_name not in df.columns:
            raise ValueError(f"Coluna '{column_name}' não existe no DataFrame. Colunas disponíveis: {df.columns}")

        # Normaliza o tipo alvo para uma instância de DataType
        if isinstance(target_type, str):
            key = target_type.lower()
            if key not in string_to_type:
                raise ValueError(f"Tipo '{target_type}' não reconhecido para a coluna '{column_name}'. "
                                 f"Tipos suportados: {list(string_to_type.keys())} ou um tipo de pyspark.sql.types.")
            spark_type = string_to_type[key]
        elif isinstance(target_type, type):
            # É uma classe de tipo (ex.: IntegerType) — instancia sem argumentos
            spark_type = target_type()
        else:
            # Já é uma instância de DataType (ex.: DecimalType(10, 2))
            spark_type = target_type

        df = df.withColumn(column_name, col(column_name).cast(spark_type))

    return df

In [0]:
# Realiza a conversão de tipo de todas as colunas, de acordo com os tipos passados. Antes de realizar a conversão, padroniza os valores inválidos, substituindo todos por NULL.
# None (NULL real) já é tratado automaticamente como inválido pela função, sem precisar estar na lista.

invalid_values = ['ERROR', 'UNKNOWN', 'NULL']

type_map = {
    'numeric_id':       'int',
    'transaction_id':   'string',
    'item':             'string',
    'quantity':         'int',
    'price_per_unit':   'double',
    'total_spent':      'double',
    'payment_method':   'string',
    'location':         'string',
    'transaction_date': 'date',
}

df = convert_column_types(df, type_map, invalid_values)

df.printSchema()

def show_invalids(df):
    """Mostra quantos valores inválidos existem em cada coluna do DataFrame"""

    print('Valores inválidos por coluna:')

    invalid_by_column = {}
    for col_name in df.columns:
        n_invalids = df.filter(df[col_name].isNull()).count()
        if n_invalids > 0:
            print(col_name, ':', n_invalids)

show_invalids(df)

# Tratamento de inválidos (padronizados como NULL)

## -> Cálculo do preço pela quantidade vendida

Verificar se há células com preço inválido, mas que pode ser calculado através da quantidade vendida e do total gasto.

In [0]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd

def plot_series_by_class(df, columns=None):
    """
    Plota um gráfico de séries temporais genérico a partir de um DataFrame Spark.

    Parâmetros:
        df     : DataFrame Spark.
        columns: dicionário com 3 chaves obrigatórias:
            'eixo_x'  -> nome da coluna usada no eixo X (ex.: 'transaction_date')
            'eixo_y'  -> nome da coluna usada no eixo Y (ex.: 'price_per_unit')
            'classe'  -> nome da coluna usada para agrupar/colorir (ex.: 'item')
    """
    if columns is None:
        columns = {'eixo_x': 'transaction_date', 'eixo_y': 'price_per_unit', 'classe': 'item'}

    col_x = columns['eixo_x']
    col_y = columns['eixo_y']
    col_class = columns['classe']

    # Coleta os dados para pandas
    pdf = df.select(col_x, col_class, col_y).toPandas()

    # Filtra linhas onde a classe é NULL (não entram na análise, pois não há como inferir o valor)
    pdf = pdf[pdf[col_class].notna()].copy()

    # Identifica quais linhas possuem valor NULL no eixo Y
    pdf['is_invalid'] = pdf[col_y].isna()

    # Filtra apenas dados válidos para o gráfico de linhas
    # Os tipos já foram convertidos anteriormente (date -> datetime, double -> float)
    pdf_valid = pdf[~pdf['is_invalid']].copy()

    # Filtra dados inválidos para plotar asteriscos
    pdf_invalid = pdf[pdf['is_invalid']].copy()

    # Mapeia cada classe ao seu valor conhecido (a partir dos dados válidos)
    class_value_map = pdf_valid.groupby(col_class)[col_y].first().to_dict()

    # Preenche o valor inválido com o valor conhecido da classe correspondente
    # (todas as classes em pdf_invalid são válidas, pois linhas com classe inválida foram removidas)
    pdf_invalid[col_y] = pdf_invalid[col_class].map(class_value_map)

    fig, ax = plt.subplots(figsize=(16, 8))

    # Plota uma linha para cada classe
    for class_name, group in pdf_valid.groupby(col_class):
        group = group.sort_values(col_x)
        ax.plot(
            group[col_x],
            group[col_y],
            marker='o',
            markersize=3,
            linestyle='-',
            linewidth=1,
            label=class_name,
            alpha=0.7
        )

    # Plota asteriscos nos pontos inválidos (que ainda têm eixo X e eixo Y conversíveis),
    # coloridos conforme a classe correspondente
    if not pdf_invalid.empty:
        # Para asteriscos, usamos apenas os que têm tanto eixo X quanto eixo Y conversíveis
        asterisk_data = pdf_invalid.dropna(subset=[col_x, col_y])
        if not asterisk_data.empty:
            # Garante que a classe seja string para o groupby
            asterisk_data = asterisk_data.copy()
            asterisk_data[col_class] = asterisk_data[col_class].astype(str)

            # Reutiliza as cores já atribuídas a cada classe nas linhas acima
            class_colors = {class_name: line.get_color()
                        for class_name, line in zip(
                            [t.get_label() for t in ax.get_lines()],
                            ax.get_lines()
                        )}

            first_class = asterisk_data[col_class].iloc[0]
            for class_name, group in asterisk_data.groupby(col_class):
                color = class_colors.get(class_name, 'red')
                ax.scatter(
                    group[col_x],
                    group[col_y],
                    marker='*',
                    s=200,
                    color=color,
                    edgecolors='black',
                    linewidths=0.5,
                    zorder=5,
                    label=f'Inválido ({col_y})' if class_name == first_class else None
                )

    ax.set_xlabel(col_x, fontsize=12)
    ax.set_ylabel(col_y, fontsize=12)
    ax.set_title(f'{col_y} por {col_class} ao longo de {col_x}', fontsize=14)
    ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

In [0]:
def fill_price_by_calculation(df):
    """
    Preenche os valores inválidos (NULL) de `price_per_unit` calculando
    a divisão de `total_spent` por `quantity`, quando ambos forem válidos.

    Parâmetros:
        df : DataFrame Spark com as colunas 'price_per_unit',
            'total_spent' e 'quantity'. Os tipos já devem estar
            convertidos e os valores inválidos já devem estar
            padronizados como NULL.

    Retorna:
        DataFrame com a coluna 'price_per_unit' atualizada.
    """
    from pyspark.sql.functions import when, col

    invalid_before = df.filter(col('price_per_unit').isNull()).count()

    df = df.withColumn(
        'price_per_unit',
        when(
            col('price_per_unit').isNull()
            & col('total_spent').isNotNull()
            & col('quantity').isNotNull()
            & (col('quantity') != 0),
            col('total_spent') / col('quantity')
        ).otherwise(col('price_per_unit'))
    )

    invalid_after = df.filter(col('price_per_unit').isNull()).count()
    print(f'Linhas com preço inválido antes: {invalid_before}')
    print(f'Linhas com preço inválido depois: {invalid_after}')

    return df

In [0]:
# A maioria das linhas com preço inválido (495/533) foram preenchidos com base na divisão do total gasto pela quantidade, já resolvendo maior parte do problema da falta de preço. As que não foram preenchidas, é devido à não possuirem dado de quantidade e/ou total gasto

display(
    df.filter(col('price_per_unit').isNull())
    )

df = fill_price_by_calculation(df)

display(
    df.filter(col('price_per_unit').isNull())
    )

## -> Preenchimento dos preços pelos produtos

Verificar se é possível preencher os valores de preço inválidos remanescentes, através da observação dos preços dos produtos.

Por exemplo, se um ítem tiver o mesmo preço ao longo de todo o período, e os inválidos estiverem nesse intervalo, pode-se inferir que aquele era o preço e corrigir os inválidos.

In [0]:
from pyspark.sql.functions import when, col, lit

def fill_invalid_prices(df, price_map):
    """
    Preenche os preços inválidos (NULL) de `price_per_unit` com base no dicionário
    `price_map` (item -> preço conhecido).

    Parâmetros:
        df        : DataFrame Spark com as colunas 'item' e 'price_per_unit'.
                    Os tipos já devem estar convertidos e os valores inválidos
                    já devem estar padronizados como NULL.
        price_map : dict mapeando item (str) -> preço conhecido.

    Retorna:
        DataFrame com a coluna 'price_per_unit' atualizada.
    """

    print('Mapeamento de preços conhecidos:')
    for item, price in sorted(price_map.items()):
        print(f'  {item}: {price}')

    # 1. Preenche os preços inválidos (NULL) com base no item correspondente
    #    Linhas com item inválido (NULL) não são alteradas
    invalid_before = df.filter(col('price_per_unit').isNull()).count()

    fill_expr = col('price_per_unit')
    for item_name, price in price_map.items():
        fill_expr = when(
            col('price_per_unit').isNull() & (col('item') == lit(item_name)),
            lit(price)
        ).otherwise(fill_expr)

    df = df.withColumn('price_per_unit', fill_expr)

    # 2. Verificação
    invalid_after = df.filter(col('price_per_unit').isNull()).count()
    print(f'\nLinhas com preço inválido antes: {invalid_before}')
    print(f'Linhas com preço inválido depois: {invalid_after}')

    return df

In [0]:
# Através do gráfico observa-se que os preços dos produtos são fixos, ou seja, não variam ao longo do tempo. Além disso, os valores inválidos não estão isolados em períodos/datas específicos, indicando não haver mudança de preços nem falta de registro em dado período, apenas falta de informação.

# Desta forma, é possível preencher os inválidos com o preço correspondente ao item. OBS: O contrário nem sempre é válido, pois há produtos com mesmo preço (smoothie e sandwich, e juice e cake), logo nem sempre é possível inferir o produto através do preço. Nesses casos serão necessárias outras análises

plot_series_columns = {
    'eixo_x': 'transaction_date',
    'eixo_y': 'price_per_unit',
    'classe': 'item'
    }

plot_series_by_class(df, plot_series_columns)

# ------------------------------------------------------------------------------------

# Pós preenchimento, verifica-se que a maioria das linhas com preço inválido (32/38) foram preenchidas, pois essas não possuem produtos associados

price_map = {
    'Coffee': 2.0,
    'Tea': 1.5,
    'Sandwich': 4.0,
    'Salad':5.0,
    'Cake': 3.0,
    'Cookie': 1.0,
    'Juice': 3.0,
    'Smoothie': 4.0,
}

df = fill_invalid_prices(df, price_map)

plot_series_by_class(df, plot_series_columns)

## -> Preenchimento dos produtos pelo preço

Verificar se é possível preencher os valores inválidos de produto, através do preço, assim como feito para preencher os preços através do produto.

In [0]:
def fill_items_by_prices(df, price_items):
    """
    Preenche os itens inválidos (NULL) de `item` com base no dicionário
    `price_items` (preço -> item conhecido) e na coluna `price_per_unit`.

    Parâmetros:
        df          : DataFrame Spark com as colunas 'item' e 'price_per_unit'.
        price_items : dict mapeando preço (float) -> item conhecido.

    Retorna:
        DataFrame com a coluna 'item' atualizada.
    """

    print('Mapeamento de preços -> itens conhecidos:')
    for price, item in sorted(price_items.items()):
        print(f'  {price}: {item}')

    # 1. Preenche os itens inválidos (NULL) com base no preço correspondente
    #    Linhas com price_per_unit inválido (NULL) não são alteradas
    invalid_before = df.filter(col('item').isNull()).count()

    fill_expr = col('item')
    for price, item_name in price_items.items():
        fill_expr = when(
            col('item').isNull()
            & col('price_per_unit').isNotNull()
            & (col('price_per_unit') == lit(price)),
            lit(item_name)
        ).otherwise(fill_expr)

    df = df.withColumn('item', fill_expr)

    # 2. Verificação
    invalid_after = df.filter(col('item').isNull()).count()
    print(f'\nLinhas com item inválido antes: {invalid_before}')
    print(f'Linhas com item inválido depois: {invalid_after}')

    return df

In [0]:
# Aqui é feita a subttituição dos itens inválidos através dos preços, ao contrário do que foi feito acima. No caso de cake e juice, e sandwich e smoothie, não é possível inferir com cereteza o preço pelo item, por isso é colocada a opção um ou outro

# Pós a substituição, verifica-se que a maioria das linhas com item inválido (963/969) foram preenchidas. As que não foram, não possuem ítens nem preços associados (Null - Null)

price_items = {
    1.0: 'Cookie',
    1.5: 'Tea',
    2.0: 'Coffee',
    3.0: 'Cake_or_Juice',
    4.0: 'Sandwich_or_Smoothie',
    5.0: 'Salad'
    }

df = fill_items_by_prices(df, price_items)

display(df.groupBy('item', 'price_per_unit').count().orderBy('item'))

## -> Preenchimento do total gasto pelo preço dos produtos x a quantidade de produtos

Com a maioria dos preços por unidade preenchidos, podemos preencher parte do total gasto pelos preços e quantidade.

In [0]:
from pyspark.sql.functions import when, col

def fill_total_spent_by_calculation(df):
    """
    Preenche os valores inválidos (NULL) de `total_spent` calculando
    a multiplicação de `price_per_unit` por `quantity`, quando ambos forem válidos.

    Parâmetros:
        df : DataFrame Spark com as colunas 'total_spent',
            'price_per_unit' e 'quantity'. Os tipos já devem estar
            convertidos e os valores inválidos já devem estar
            padronizados como NULL.

    Retorna:
        DataFrame com a coluna 'total_spent' atualizada.
    """
    invalid_before = df.filter(col('total_spent').isNull()).count()

    df = df.withColumn(
        'total_spent',
        when(
            col('total_spent').isNull()
            & col('price_per_unit').isNotNull()
            & col('quantity').isNotNull(),
            col('price_per_unit') * col('quantity')
        ).otherwise(col('total_spent'))
    )

    invalid_after = df.filter(col('total_spent').isNull()).count()
    print(f'Linhas com total_spent inválido antes: {invalid_before}')
    print(f'Linhas com total_spent inválido depois: {invalid_after}')

    return df
    

In [0]:
# Verifica-se que a maioria das linhas com total_spent inválido (481/502) foram preenchidas
 
display(
    df.filter(col('total_spent').isNull())
)

df = fill_total_spent_by_calculation(df)

display(
    df.filter(col('total_spent').isNull())
)

## -> Preenchimento da quantidade vendida pelo total gasto / pelo preço dos produtos

Com a maioria dos preços por unidade preenchidos e o total gasto preenchido, podemos preencher parte da quantidade vendida

In [0]:
from pyspark.sql.functions import when, col

def fill_quantity_by_calculation(df):
    """
    Preenche os valores inválidos (NULL) de `quantity` calculando
    a divisão de `total_spent` por `price_per_unit`, quando ambos forem válidos.

    Parâmetros:
        df : DataFrame Spark com as colunas 'quantity',
            'total_spent' e 'price_per_unit'. Os tipos já devem estar
            convertidos e os valores inválidos já devem estar
            padronizados como NULL.

    Retorna:
        DataFrame com a coluna 'quantity' atualizada.
    """
    invalid_before = df.filter(col('quantity').isNull()).count()

    df = df.withColumn(
        'quantity',
        when(
            col('quantity').isNull()
            & col('total_spent').isNotNull()
            & col('price_per_unit').isNotNull()
            & (col('price_per_unit') != 0),
            col('total_spent') / col('price_per_unit')
        ).otherwise(col('quantity'))
    )

    invalid_after = df.filter(col('quantity').isNull()).count()
    print(f'Linhas com quantity inválido antes: {invalid_before}')
    print(f'Linhas com quantity inválido depois: {invalid_after}')

    return df


In [0]:
# Verifica-se que a maioria das linhas com total_spent inválido (481/502) foram preenchidas
 
display(
    df.filter(col('quantity').isNull())
)

df = fill_quantity_by_calculation(df)

display(
    df.filter(col('quantity').isNull())
)

## -> Tentativa de preenchimento de datas inválidas

In [0]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd

def plot_temporal_series(df, col_x, col_y):
    """
    Plota um gráfico de dispersão (scatter) de uma série temporal,
    ordenando o eixo do tempo (col_x).

    Parâmetros:
        df     : DataFrame Spark.
        col_x  : nome da coluna usada no eixo X (tempo). Deve ser do tipo date/timestamp.
        col_y  : nome da coluna usada no eixo Y (valor a ser analisado).
    """
    # Coleta os dados para pandas mantendo apenas as duas colunas de interesse
    pdf = df.select(col_x, col_y).toPandas()

    # Remove linhas em que o eixo X (tempo) é nulo — sem data não há como ordenar
    pdf = pdf.dropna(subset=[col_x]).copy()

    # Ordena pelo eixo do tempo
    pdf = pdf.sort_values(col_x)

    fig, ax = plt.subplots(figsize=(16, 8))

    ax.plot(
        pdf[col_x],
        pdf[col_y],
    )

    ax.set_xlabel(col_x, fontsize=12)
    ax.set_ylabel(col_y, fontsize=12)
    ax.set_title(f'{col_y} ao longo de {col_x}', fontsize=14)

    # Tenta usar formatador de data no eixo X; se a coluna for datetime, aplica rotação
    try:
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
    except Exception:
        pass

    plt.xticks(rotation=45)
    #plt.yscale('log')
    plt.tight_layout()
    plt.show()

In [0]:
def plot_temporal_series_by_average(df, col_x, col_y):
    """
    Plota um gráfico de dispersão (scatter) de uma série temporal,
    ordenando o eixo do tempo (col_x).

    Parâmetros:
        df     : DataFrame Spark.
        col_x  : nome da coluna usada no eixo X (tempo). Deve ser do tipo date/timestamp.
        col_y  : nome da coluna usada no eixo Y (valor a ser analisado).
    """
    # Coleta os dados para pandas mantendo apenas as duas colunas de interesse
    pdf = df.select(col_x, col_y).toPandas()

    # Remove linhas em que o eixo X (tempo) é nulo — sem data não há como ordenar
    pdf = pdf.dropna(subset=[col_x]).copy()

    # Agrupa os valores pelo eixo do tempo, calculando a média por data
    pdf = pdf.groupby(col_x, as_index=False)[col_y].mean()

    # Ordena pelo eixo do tempo
    pdf = pdf.sort_values(col_x)

    fig, ax = plt.subplots(figsize=(16, 8))

    ax.plot(
        pdf[col_x],
        pdf[col_y],
    )

    ax.set_xlabel(col_x, fontsize=12)
    ax.set_ylabel(col_y, fontsize=12)
    ax.set_title(f'Average {col_y} ao longo de {col_x}', fontsize=14)

    # Tenta usar formatador de data no eixo X; se a coluna for datetime, aplica rotação
    try:
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
    except Exception:
        pass

    plt.xticks(rotation=45)
    #plt.yscale('log')
    plt.tight_layout()
    plt.show()

In [0]:
# Plotando diretamente o ID ao longo do tempo, vefica-se que não há uma tendência ou agrupamento dos ids por data, logo não são ordenados.

# E ao plotar a média dos IDs (agrupado por dia), verifica-se o mesmo. Ou seja, a tentativa de preencher as datas inválidas através da tendência de crescimento do ID não é uma estratégia adequada para esses dados.
 
col_x = 'transaction_date'
col_y = 'numeric_id'

plot_temporal_series(df, col_x, col_y)

plot_temporal_series_by_average(df, col_x, col_y)

## -> Tentativa de associação entre método de pagamento, local de compra e produto

In [0]:
import matplotlib.pyplot as plt
import pandas as pd

def plot_histogram_by_combination(df, col1, col2):
    """
    Plota um histograma (gráfico de barras) onde as classes do eixo X são os
    valores de col1, e as subclasses de col2 são representadas por cores
    diferentes na legenda, ignorando linhas onde qualquer uma delas é NULL.

    Parâmetros:
        df   : DataFrame Spark.
        col1 : nome da primeira coluna (ex.: 'location').
        col2 : nome da segunda coluna (ex.: 'payment_method').
    """
    # Coleta apenas as duas colunas de interesse
    pdf = df.select(col1, col2).toPandas()

    # Remove linhas em que qualquer coluna é NULL
    pdf = pdf.dropna(subset=[col1, col2]).copy()

    # Conta ocorrências de cada combinação e reorganiza em tabela cruzada
    crosstab = pd.crosstab(pdf[col1], pdf[col2])

    categories = crosstab.index.tolist()
    subcategories = crosstab.columns.tolist()
    n_subs = len(subcategories)

    fig, ax = plt.subplots(figsize=(14, 7))

    bar_width = 0.8 / max(n_subs, 1)
    x_base = range(len(categories))

    for j, sub in enumerate(subcategories):
        offsets = [x + j * bar_width for x in x_base]
        values = crosstab[sub].values
        bars = ax.bar(offsets, values, width=bar_width, edgecolor='black',
                      alpha=0.8, label=str(sub))
        # Rótulos nas barras
        for bar in bars:
            height = bar.get_height()
            if height > 0:
                ax.text(bar.get_x() + bar.get_width() / 2, height,
                        str(int(height)), ha='center', va='bottom', fontsize=8)

    ax.set_xlabel(col1, fontsize=12)
    ax.set_ylabel('Contagem', fontsize=12)
    ax.set_title(f'Histograma de {col1} por {col2}', fontsize=14)
    ax.set_xticks([x + bar_width * (n_subs - 1) / 2 for x in x_base])
    ax.set_xticklabels(categories, rotation=45, ha='right')
    ax.legend(title=col2, bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)

    plt.tight_layout()
    plt.show()

In [0]:
# Verifica-se que não há uma tendência de método de pagamento dependendo da localização, ou seja, se é uma compra em loja física ou pedido on-line.

# Da mesma forma, não é verificada uma relação entre método de pagamento e local da compra com os produtos. Ou seja, não há um ou mais produtos comprados preferencialmente através de um método específico de pagamento ou local de compra.

# Logo não é recomendado preencher os campos de método de pagamento e local da compra baseados na relaçãp entre essas variáveis, nem com base nos ítens (produtos)

col1 = 'location'
col2 = 'payment_method'
plot_histogram_by_combination(df, col1, col2)

col1 = 'item'
col2 = 'payment_method'
plot_histogram_by_combination(df, col1, col2)

col1 = 'item'
col2 = 'location'
plot_histogram_by_combination(df, col1, col2)

## -> Verificação final dos resultados

In [0]:
# Verifica-se que, após a limpeza, a quantidade de dados de item, quantity, price_per_unity e total_spent foi quase que inteiramente preenchida.

# Com base em análises realizadas, não há evidências de que se possa preencher  o método de pagamento, a localização (tipo) de compra e a data com as estratégias utilizadas. Para tal, recomenda-se deixar como desconhecido/vazio e realizar análises mais aprofundades, por exemplo testagem de algoritmos de Machine Learning.

show_invalids(df)

## -> Salvamento do dataset tratado

In [0]:
# Parâmetros de entrada
output_database = 'my_database'
output_table_name = 'clean_cafe_sales_table'

# Salvamento
output_table = f'workspace.{output_database}.{output_table_name}'

df.write \
    .option('overwriteSchema', 'true') \
    .saveAsTable(
        name=output_table,
        mode='overwrite',
        partitionBy=None
    )

print(f"\nTabela criada com sucesso no Catalog, em: {output_table}")